In [1]:
import os
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
from scipy import stats
from calculate_scnetwork_precision_recall import calculate_scnetwork_precision_recall

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [2]:
DATASET = "retinal"
BASE    = f"../../../data/processed/{DATASET}"

rna      = ad.read_h5ad(f"output/{DATASET}_with_networks.h5ad")
oob_df   = rna.obsm["wScReNI_oob_r2"]
cell_r2  = oob_df.mean(axis=1)
gene_r2  = oob_df.mean(axis=0)
genes    = rna.var_names.tolist()

In [3]:
data         = np.load(f"output/{DATASET}_multirun.npz")
all_networks = data["networks"]
all_oob      = data["oob_scores"]
seeds        = data["seeds"]

print(f"Networks shape : {all_networks.shape}")
print(f"OOB shape      : {all_oob.shape}")
print(f"Seeds used     : {seeds}")

Networks shape : (5, 400, 500, 500)
OOB shape      : (5, 400, 500)
Seeds used     : [ 892 7739 6545 4388 4330]


In [4]:
ground_truth = pd.read_csv(
    "../../../data/processed/mmp9.TSV.5kb_TF_target.df.txt",
    sep="\t"
)

genes_in_network = set(genes)
ground_truth_filtered = ground_truth[
    ground_truth["TF"].isin(genes_in_network) &
    ground_truth["Target_genes"].isin(genes_in_network)
]
tf_target_reference = (
    ground_truth_filtered["Target_genes"].astype(str)
    + "_"
    + ground_truth_filtered["TF"].astype(str)
).tolist()

print(f"Filtered gold standard pairs: {len(tf_target_reference)}")

Filtered gold standard pairs: 5896


In [5]:
def build_networks_df(networks_array, gene_names):
    return [
        pd.DataFrame(networks_array[i], index=gene_names, columns=gene_names)
        for i in range(len(networks_array))
    ]

In [6]:
all_pr = []

for i in range(5):
    print(f"Computing precision/recall for run {i+1}/5...")
    networks_df = build_networks_df(all_networks[i], genes)

    scNetworks = {
        "CSN":     networks_df,
        "wScReNI": networks_df,
    }

    results = calculate_scnetwork_precision_recall(
        scNetworks     = scNetworks,
        TF_target_pair = tf_target_reference,
        top_number     = (0,),
    )

    df_run = results[0][results[0]["scNetwork_type"] == "wScReNI"].copy()
    df_run = df_run.reset_index(drop=True)
    df_run["run"]    = i
    df_run["seed"]   = seeds[i]
    df_run["rf_r2"]  = cell_r2.values
    all_pr.append(df_run)

Computing precision/recall for run 1/5...
Computing precision/recall for run 2/5...
Computing precision/recall for run 3/5...
Computing precision/recall for run 4/5...
Computing precision/recall for run 5/5...


In [7]:
pr_df = pd.concat(all_pr, ignore_index=True)
print(f"\nPrecision/recall DataFrame shape: {pr_df.shape}")
print(pr_df.head(10))



Precision/recall DataFrame shape: (2000, 6)
  scNetwork_type  precision    recall  run  seed     rf_r2
0        wScReNI   0.024897  0.125509    0   892 -0.100838
1        wScReNI   0.019862  0.077001    0   892 -0.147914
2        wScReNI   0.023052  0.107022    0   892 -0.120601
3        wScReNI   0.020879  0.080054    0   892 -0.140592
4        wScReNI   0.026872  0.176730    0   892 -0.104271
5        wScReNI   0.021935  0.102782    0   892 -0.109150
6        wScReNI   0.022409  0.085991    0   892 -0.141140
7        wScReNI   0.034950  0.180122    0   892 -0.117648
8        wScReNI   0.022368  0.103969    0   892 -0.092841
9        wScReNI   0.044343  0.259159    0   892 -0.108592


In [8]:
edge_variance     = all_networks.var(axis=0)
mean_edge_var     = edge_variance.mean(axis=(1, 2))

print(f"\nEdge variance shape : {edge_variance.shape}")
print(f"Mean edge var shape : {mean_edge_var.shape}")
print(pd.Series(mean_edge_var).describe())



Edge variance shape : (400, 500, 500)
Mean edge var shape : (400,)
count    4.000000e+02
mean     1.697048e-08
std      2.228741e-09
min      1.179627e-08
25%      1.578996e-08
50%      1.718189e-08
75%      1.843526e-08
max      2.462672e-08
dtype: float64


In [9]:
mean_pr = pr_df.groupby(pr_df.index % 400)[["precision", "recall"]].mean()

print("\nCorrelations with OOB R²:")
for metric in ["precision", "recall"]:
    r, p = stats.spearmanr(cell_r2.values, mean_pr[metric])
    print(f"  {metric:10s}: r={r:.4f}, p={p:.2e}")

print("\nCorrelations with mean edge variance:")
for metric in ["precision", "recall"]:
    r, p = stats.spearmanr(mean_edge_var, mean_pr[metric])
    print(f"  {metric:10s}: r={r:.4f}, p={p:.2e}")



Correlations with OOB R²:
  precision : r=0.0466, p=3.53e-01
  recall    : r=0.4082, p=1.70e-17

Correlations with mean edge variance:
  precision : r=0.5157, p=1.45e-28
  recall    : r=0.6630, p=5.43e-52


In [10]:
pr_df.to_csv(f"output/{DATASET}_multirun_precision_recall.csv", index=False)
np.save(f"output/{DATASET}_edge_variance.npy", mean_edge_var)

print(f"\nSaved:")
print(f"  output/{DATASET}_multirun_precision_recall.csv")
print(f"  output/{DATASET}_edge_variance.npy")


Saved:
  output/retinal_multirun_precision_recall.csv
  output/retinal_edge_variance.npy
